# Energy Forecast — Prediction Performance (last 30 days)

Pulls live data from Home Assistant via SMB, fetches weather from Open-Meteo, and renders accuracy charts for the last 30 days.

**Pre-requisite:** `SMB_PASSWORD` environment variable must be set before starting the kernel.


In [ ]:
from __future__ import annotations

import io
import json
import os
from datetime import date, timedelta

import altair as alt
import pandas as pd
import requests
import yaml
from smb.SMBConnection import SMBConnection


In [ ]:
# ── Credentials ──────────────────────────────────────────────────────────────
SMB_USER = os.getenv("SMB_USER", "martin")
SMB_PASSWORD = os.getenv("SMB_PASSWORD")
if not SMB_PASSWORD:
    raise RuntimeError(
        "SMB_PASSWORD environment variable is not set. "
        "Set it before starting the kernel: export SMB_PASSWORD=<password>"
    )

# ── Constants ─────────────────────────────────────────────────────────────────
HA_HOST = "homeassistant"
SMB_SHARE = "addon_configs"
AD_BASE = "a0d7b954_appdaemon/apps"
FORECAST_REMOTE = f"{AD_BASE}/energy_forecast"

TZ = "Europe/Zurich"
CUTOFF_DAYS = 30
EV_THRESHOLD_KWH = 7.0  # matches EV_CHARGING_THRESHOLD_KWH in const.py

WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

print("Config OK — SMB_PASSWORD is set")

## 1 · Fetch data from Home Assistant (SMB)

In [ ]:
def _smb_read(remote_path: str) -> bytes:
    conn = SMBConnection(SMB_USER, SMB_PASSWORD, "notebook", HA_HOST, use_ntlm_v2=True)
    if not conn.connect(HA_HOST, 445):
        raise ConnectionError(f"SMB connection to {HA_HOST}:445 failed")
    try:
        buf = io.BytesIO()
        conn.retrieveFile(SMB_SHARE, remote_path, buf)
        return buf.getvalue()
    finally:
        conn.close()


# Pull files
print("Fetching pred_history.json …")
_pred_raw = json.loads(_smb_read(f"{FORECAST_REMOTE}/pred_history.json"))
print(f"  pred entries : {len(_pred_raw.get('pred', {}))}")
print(f"  actual entries: {len(_pred_raw.get('actuals', {}))}")

print("Fetching apps.yaml …")
_apps_yaml = yaml.safe_load(_smb_read(f"{AD_BASE}/apps.yaml"))

if not _pred_raw.get("pred"):
    print("WARNING: pred_history.json has 0 prediction entries — charts will be empty.")
    print("The app needs at least one update cycle on the HA system to populate this file.")


## 2 · Fetch outdoor temperature from Open-Meteo archive

Uses the Open-Meteo historical archive API to download hourly 2 m air temperature for the same date range as the prediction data. Latitude/longitude are read from `apps.yaml` (the `energy_forecast` section) and fall back to Zurich city centre if not present. The result is stored as `temp_df` and is used later for the temperature-vs-error correlation chart.

In [ ]:
# Read lat/lon from apps.yaml; fall back to Zurich city centre
_ef_cfg = _apps_yaml.get("energy_forecast", {})
_lat = _ef_cfg.get("open_meteo_latitude", 47.376)
_lon = _ef_cfg.get("open_meteo_longitude", 8.541)

_start = (date.today() - timedelta(days=CUTOFF_DAYS)).isoformat()
_end = date.today().isoformat()

_url = (
    f"https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={_lat}&longitude={_lon}"
    f"&start_date={_start}&end_date={_end}"
    f"&hourly=temperature_2m&timezone=Europe%2FZurich"
)

try:
    _resp = requests.get(_url, timeout=30)
    _resp.raise_for_status()
    _weather = _resp.json()
    temp_df = pd.DataFrame({
        "timestamp": pd.to_datetime(_weather["hourly"]["time"]),
        "temp_c": _weather["hourly"]["temperature_2m"],
    })
    temp_df["timestamp"] = temp_df["timestamp"].dt.floor("h")
    print(f"Weather fetched: {len(temp_df)} hourly rows, {_start} → {_end}")
except Exception as exc:
    temp_df = pd.DataFrame(columns=["timestamp", "temp_c"])
    print(f"WARNING: Open-Meteo fetch failed ({exc}). Chart 6 (temp correlation) will be skipped.")
